# diffBloch Event Report

This notebook renders figures from the canonical `ReportLogger` JSONL event stream. The refinement library does not write visualization images; optional figure export happens here only.

Point it at a report by setting `EVENT_LOG`, launching Jupyter with `DIFFBLOCH_EVENT_LOG=/path/to/report.jsonl`, editing the path text box below, or dropping a JSONL file into the upload control when `ipywidgets` is available.

In [ ]:
from __future__ import annotations

import math
import os
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display

from diffBloch.observability import EventRecord

try:
    import ipywidgets as widgets
except ModuleNotFoundError:
    widgets = None


def repository_root(start: Path | None = None) -> Path:
    current = (Path.cwd() if start is None else start).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "diffBloch").is_dir():
            return candidate
    return Path.cwd().resolve()


def resolve_event_log_path(path: Path | str) -> Path:
    candidate = Path(path).expanduser()
    if candidate.is_absolute():
        if candidate.is_file():
            return candidate
        raise FileNotFoundError(f"Event log does not exist: {candidate}")

    roots = (Path.cwd().resolve(), repository_root())
    attempts = []
    for root in roots:
        resolved = root / candidate
        if resolved not in attempts:
            attempts.append(resolved)
        if resolved.is_file():
            return resolved
    tried = "\n  ".join(str(attempt) for attempt in attempts)
    raise FileNotFoundError(f"Event log not found. Tried:\n  {tried}")


def default_event_log() -> Path:
    root = repository_root()
    reports = sorted(
        (
            *Path("reproducibility/reports").glob("report-*.jsonl"),
            *Path("reproducibility").glob("report-*.jsonl"),
            *root.glob("examples/*/data/*/reproducibility/reports/report-*.jsonl"),
            *root.glob("examples/*/data/*/reproducibility/report-*.jsonl"),
        ),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    if reports:
        return reports[0]
    return Path("reproducibility/events.jsonl")


EVENT_LOG = Path(os.environ.get("DIFFBLOCH_EVENT_LOG", str(default_event_log())))
EXPORT_FIGURES = False
EXPORT_DIR = Path("event_report_figures")
EXPORT_FORMATS = ("svg",)

In [ ]:
try:
    _existing_widgets = widgets
except NameError:
    import math
    import os
    from collections import defaultdict
    from pathlib import Path

    import matplotlib.pyplot as plt
    from IPython.display import display

    from diffBloch.observability import EventRecord

    try:
        import ipywidgets as widgets
    except ModuleNotFoundError:
        widgets = None

    def repository_root(start: Path | None = None) -> Path:
        current = (Path.cwd() if start is None else start).resolve()
        for candidate in (current, *current.parents):
            if (candidate / "pyproject.toml").is_file() and (
                candidate / "src" / "diffBloch"
            ).is_dir():
                return candidate
        return Path.cwd().resolve()

    def resolve_event_log_path(path: Path | str) -> Path:
        candidate = Path(path).expanduser()
        if candidate.is_absolute():
            if candidate.is_file():
                return candidate
            raise FileNotFoundError(f"Event log does not exist: {candidate}")

        roots = (Path.cwd().resolve(), repository_root())
        attempts = []
        for root in roots:
            resolved = root / candidate
            if resolved not in attempts:
                attempts.append(resolved)
            if resolved.is_file():
                return resolved
        tried = "\n  ".join(str(attempt) for attempt in attempts)
        raise FileNotFoundError(f"Event log not found. Tried:\n  {tried}")

    def default_event_log() -> Path:
        root = repository_root()
        reports = sorted(
            (
                *Path("reproducibility/reports").glob("report-*.jsonl"),
                *Path("reproducibility").glob("report-*.jsonl"),
                *root.glob("examples/*/data/*/reproducibility/reports/report-*.jsonl"),
                *root.glob("examples/*/data/*/reproducibility/report-*.jsonl"),
            ),
            key=lambda path: path.stat().st_mtime,
            reverse=True,
        )
        if reports:
            return reports[0]
        return Path("reproducibility/events.jsonl")

    EVENT_LOG = Path(os.environ.get("DIFFBLOCH_EVENT_LOG", str(default_event_log())))
    EXPORT_FIGURES = False
    EXPORT_DIR = Path("event_report_figures")
    EXPORT_FORMATS = ("svg",)


event_log_text = None
event_log_upload = None
if widgets is not None:
    event_log_text = widgets.Text(
        value=str(EVENT_LOG),
        description="JSONL",
        layout=widgets.Layout(width="80%"),
    )
    event_log_upload = widgets.FileUpload(
        accept=".jsonl,application/jsonl,application/x-ndjson",
        multiple=False,
        description="Upload JSONL",
    )
    display(event_log_text, event_log_upload)
else:
    print(f"Using EVENT_LOG={EVENT_LOG}. Install ipywidgets for upload/drag-drop controls.")


def read_records(path: Path) -> list[EventRecord]:
    resolved = resolve_event_log_path(path)
    return [
        EventRecord.model_validate_json(line)
        for line in resolved.read_text().splitlines()
        if line.strip()
    ]


def read_records_text(text: str) -> list[EventRecord]:
    return [EventRecord.model_validate_json(line) for line in text.splitlines() if line.strip()]


def uploaded_text(upload) -> str | None:
    if upload is None or not upload.value:
        return None
    value = upload.value
    item = next(iter(value.values())) if isinstance(value, dict) else value[0]
    content = item["content"]
    return bytes(content).decode("utf-8")


def selected_event_log() -> Path:
    if event_log_text is not None:
        return Path(event_log_text.value).expanduser()
    return EVENT_LOG.expanduser()


def read_selected_records() -> list[EventRecord]:
    text = uploaded_text(event_log_upload)
    if text is not None:
        return read_records_text(text)
    return read_records(selected_event_log())


def records_of(records: list[EventRecord], event_type: str) -> list[EventRecord]:
    return [record for record in records if record.event_type == event_type]


def finite(values):
    return [float(value) for value in values if value is not None and math.isfinite(float(value))]


records = read_selected_records()
len(records)

In [ ]:
def plot_epoch_curve(records: list[EventRecord]):
    steps = records_of(records, "RefinementStep")
    if not steps:
        return None
    x = [record.payload["iteration"] + 1 for record in steps]
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for key, label in (
        ("wr2", "train wR2"),
        ("r_obs", "train R_obs"),
        ("val_wr2", "validation wR2"),
        ("val_r_obs", "validation R_obs"),
    ):
        y = [record.payload.get(key) for record in steps]
        if any(value is not None for value in y):
            ax.plot(x, y, marker="o", linewidth=1.5, label=label)
    ax.set_xlabel("epoch")
    ax.set_ylabel("score")
    ax.set_title("Epoch curve")
    ax.grid(True, alpha=0.25)
    ax.legend()
    return fig


def plot_orientation_optimization(records: list[EventRecord]):
    fits = records_of(records, "OrientationOptimized")
    if not fits:
        return None
    fits = sorted(fits, key=lambda record: (record.dataset or "", record.rotation_index or -1))
    x = range(len(fits))
    labels = [f"{record.dataset or ''}:{record.rotation_index}" for record in fits]
    fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
    after = [record.payload.get("score") for record in fits]
    before = [record.payload.get("seed_score") for record in fits]
    axes[0].plot(x, before, marker="o", linewidth=1, label="before")
    axes[0].plot(x, after, marker="o", linewidth=1, label="after")
    axes[0].set_ylabel("score")
    axes[0].set_title("Orientation optimization")
    axes[0].grid(True, alpha=0.25)
    axes[0].legend()
    for key, label in (("alpha", "alpha"), ("beta", "beta"), ("omega", "omega")):
        axes[1].plot(
            x, [record.payload.get(key) for record in fits], marker="o", linewidth=1, label=label
        )
    axes[1].set_ylabel("delta angle (deg)")
    axes[1].set_xticks(list(x)[:: max(1, len(fits) // 12)])
    axes[1].set_xticklabels(labels[:: max(1, len(fits) // 12)], rotation=45, ha="right")
    axes[1].grid(True, alpha=0.25)
    axes[1].legend()
    fig.tight_layout()
    return fig


def plot_dataset_summary(records: list[EventRecord]):
    metrics = [record for record in records_of(records, "RefinedRotationMetrics") if record.dataset]
    datasets = sorted({record.dataset for record in metrics})
    if len(datasets) <= 1:
        return None
    grouped = defaultdict(list)
    for record in metrics:
        grouped[record.dataset].append(record)
    wr2 = [
        sum(finite(record.payload.get("wr2") for record in grouped[dataset]))
        / len(finite(record.payload.get("wr2") for record in grouped[dataset]))
        for dataset in datasets
    ]
    r_obs = [
        sum(finite(record.payload.get("r_obs") for record in grouped[dataset]))
        / len(finite(record.payload.get("r_obs") for record in grouped[dataset]))
        for dataset in datasets
    ]
    x = range(len(datasets))
    fig, ax = plt.subplots(figsize=(8, 4.5))
    width = 0.38
    ax.bar([value - width / 2 for value in x], wr2, width=width, label="wR2")
    ax.bar([value + width / 2 for value in x], r_obs, width=width, label="R_obs")
    ax.set_xticks(list(x))
    ax.set_xticklabels(datasets, rotation=30, ha="right")
    ax.set_title("Per-dataset final scores")
    ax.legend()
    fig.tight_layout()
    return fig


def plot_thickness_grids(records: list[EventRecord]):
    fits = records_of(records, "ThicknessOptimized")
    if not fits:
        return None
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for record in fits:
        x = record.series.get("candidate_thicknesses")
        y = record.series.get("candidate_score")
        if x and y:
            ax.plot(x, y, linewidth=0.8, alpha=0.45)
    ax.set_ylim(bottom=0)
    ax.set_xlabel("thickness")
    ax.set_ylabel("score")
    ax.set_title("Thickness score grids")
    ax.grid(True, alpha=0.25)
    return fig


figures = {
    "epoch_curve": plot_epoch_curve(records),
    "orientation_optimization": plot_orientation_optimization(records),
    "per_dataset_summary": plot_dataset_summary(records),
    "thickness_grids": plot_thickness_grids(records),
}
figures = {name: fig for name, fig in figures.items() if fig is not None}
figures.keys()

In [ ]:
def export_figures(
    figures: dict[str, plt.Figure], output_dir: Path, formats=("svg",)
) -> list[Path]:
    output_dir.mkdir(parents=True, exist_ok=True)
    written = []
    for name, fig in figures.items():
        for fmt in formats:
            path = output_dir / f"{name}.{fmt}"
            fig.savefig(path, bbox_inches="tight", dpi=160)
            written.append(path)
    return written


written = export_figures(figures, EXPORT_DIR, EXPORT_FORMATS) if EXPORT_FIGURES else []
written